# Вариант 7
Рассмотрим систему диффиренциальных уравнений:

$$
\begin{pmatrix} \dot{x} \\ \dot{y} \\ \dot{z} \end{pmatrix} = \begin{pmatrix} 0 & 0 & -1 \\ 0 & -1 & 0 \\ 1.7 & 1 & 0 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ -x^2 \\ 1.7 \end{pmatrix}
$$

**Цели работы:**

- Найти стационарные точки системы, описываемой следующей автономной системой обыкновенных дифференциальных уравнений, исследовать их устойчивость.
- Построить фазовый портрет.
- Построить сечение Пуанкаре (одностороннее и двустороннее) аттрактора динамической системы.
- Построить график расхождения двух близлежащих на аттрактора траекторий (решить систему с двумя близкими начальными условиями и построить график нормы разности между решениями в зависимости от времени).
- Найти спектр показателей Ляпунова аттрактора.

### Найти стационарные точки системы, описываемой следующей автономной системой обыкновенных дифференциальных уравнений, исследовать их устойчивость.

In [46]:
from sympy import *

def is_stable(J):
    eig_dict = J.eigenvals()
    for val, mult in eig_dict.items():
        if re(val) >= 0:
            return False
    return True

x, y, z = symbols('x y z')

dx = -z
dy = -y - x**2
dz = 1.7*x + y + 1.7

solutions = solve([dx, dy, dz], [x, y, z])

J = Matrix([
    [diff(dx, x), diff(dx, y), diff(dx, z)],
    [diff(dy, x), diff(dy, y), diff(dy, z)],
    [diff(dz, x), diff(dz, y), diff(dz, z)],
])

for i, sol in enumerate(solutions, start=1):
    J_sol = J.subs({x: sol[0], y: sol[1], z: sol[2]})
    eig_dict = J_sol.eigenvals()
    eig_pretty = ", ".join([f"{N(val, 6)} (mult={mult})" for val, mult in eig_dict.items()])

    print(f"Point {i}: {sol}")
    print(f"Eigenvalues: {eig_pretty}")
    print(f"Is stable: {is_stable(J_sol)}")
    display(J_sol)
    print()

Point 1: (-0.706438241627338, -0.499054989233525, 0.0)
Eigenvalues: 0.194614 - 1.4842*I (mult=1), -1.38923 - 4.17778e-64*I (mult=1), 0.194614 + 1.4842*I (mult=1)
Is stable: False


Matrix([
[               0,  0, -1],
[1.41287648325468, -1,  0],
[             1.7,  1,  0]])


Point 2: (2.40643824162734, -5.79094501076647, 0.0)
Eigenvalues: -0.953683 - 1.58782*I (mult=1), 0.907365 + 7.68246e-64*I (mult=1), -0.953683 + 1.58782*I (mult=1)
Is stable: False


Matrix([
[                0,  0, -1],
[-4.81287648325468, -1,  0],
[              1.7,  1,  0]])

In [47]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

A = 1.7
DEFAULT_RTOL = 1e-9
DEFAULT_ATOL = 1e-11

STATIC_POINTS = np.array([
    [-0.706438241627338, -0.499054989233525, 0.0],
    [2.40643824162734, -5.79094501076647, 0.0],
])
STATIC_LABELS = ["P1", "P2"]


def rhs(s):
    x, y, z = s
    return np.asarray([
        -z,
        -y - x**2,
        A * x + y + A,
    ])


def f(t, s):
    return rhs(s)


def jacobian(s):
    x, y, z = s
    return np.array([
        [0.0, 0.0, -1.0],
        [-2.0 * x, -1.0, 0.0],
        [A, 1.0, 0.0],
    ], dtype=float)

### Построить фазовый портрет.

In [48]:
from plotly.colors import sample_colorscale

# Segment duration in seconds
dt = 0.1

# Two points to highlight
points_to_mark = [tuple(p) for p in STATIC_POINTS]

# One trajectory settings
x0, y0, z0 = 0.0, 0.0, 0.0
trajectory_start = (x0, y0, z0)
t_span = (0.0, 40.0)
t_eval = np.linspace(t_span[0], t_span[1], 2200)

# System dynamics are defined in the general cell above.

# Build a 3D grid for short line segments
xg = np.linspace(-4.0, 4.0, 9)
yg = np.linspace(-8.0, 2.0, 9)
zg = np.linspace(-2.0, 2.0, 7)
X, Y, Z = np.meshgrid(xg, yg, zg, indexing="ij")

DX, DY, DZ = rhs((X, Y, Z))
S = np.sqrt(DX**2 + DY**2 + DZ**2)  # speed

# Endpoints after dt seconds (Euler step)
X2 = X + dt * DX
Y2 = Y + dt * DY
Z2 = Z + dt * DZ

# Colors by speed (same mapping for segments and arrows)
s_min = float(S.min())
s_max = float(S.max())
if s_max - s_min < 1e-12:
    s_norm = np.zeros_like(S)
else:
    s_norm = (S - s_min) / (s_max - s_min)

fig = go.Figure()

# Draw one colored short segment per grid point
for x1, y1, z1, x2, y2, z2, sn in zip(
    X.ravel(), Y.ravel(), Z.ravel(),
    X2.ravel(), Y2.ravel(), Z2.ravel(),
    s_norm.ravel(),
):
    color = sample_colorscale("Viridis", [float(sn)])[0]
    fig.add_trace(
        go.Scatter3d(
            x=[x1, x2],
            y=[y1, y2],
            z=[z1, z2],
            mode="lines",
            line=dict(width=4, color=color),
            showlegend=False,
        )
    )

fig.add_trace(
    go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode="markers",
        marker=dict(
            size=0.1,
            color=[s_min, s_max],
            cmin=s_min,
            cmax=s_max,
            colorscale="Viridis",
            colorbar=dict(title="speed"),
            showscale=True,
        ),
        name="speed",
        showlegend=True,
        hoverinfo="skip",
    )
)

sol = solve_ivp(f, t_span, trajectory_start, t_eval=t_eval, rtol=1e-8, atol=1e-10)
fig.add_trace(
    go.Scatter3d(
        x=sol.y[0],
        y=sol.y[1],
        z=sol.y[2],
        mode="lines",
        line=dict(width=5, color="black"),
        name="trajectory",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=[p[0] for p in points_to_mark],
        y=[p[1] for p in points_to_mark],
        z=[p[2] for p in points_to_mark],
        mode="markers+text",
        marker=dict(size=8, color=["red", "orange"], symbol="diamond"),
        text=["P1", "P2"],
        textposition="top center",
        name="marked points",
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="cube",
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.2)),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

### Построить сечение Пуанкаре аттрактора динамической системы.

In [49]:
# Uses shared imports and system function from the general cell.

# -------- Parameters --------
section_z = 0.0              # Poincare section: z = 0
s0 = [0.2, -0.2, 0.1]        # Initial condition (not exactly on section)
t_span = (0.0, 600.0)        # Long integration for attractor sampling
discard_time = 120.0         # Transient to skip
rtol, atol = DEFAULT_RTOL, DEFAULT_ATOL

# Two static reference points
static_points = STATIC_POINTS
static_labels = STATIC_LABELS

# -------- Integrate with dense output --------
sol = solve_ivp(
    f,
    t_span,
    s0,
    dense_output=True,
    max_step=0.05,
    rtol=rtol,
    atol=atol,
)

if not sol.success:
    raise RuntimeError(f"Integration failed: {sol.message}")

# -------- Detect crossings of z-section --------
t_grid = sol.t
xyz_grid = sol.y.T  # columns: x, y, z

cross_all = []   # two-sided crossings
cross_up = []    # one-sided crossings (z goes from - to +)
cross_dn = []    # opposite direction (for coloring in two-sided plot)

for i in range(len(t_grid) - 1):
    t1, t2 = t_grid[i], t_grid[i + 1]
    z1 = xyz_grid[i, 2] - section_z
    z2 = xyz_grid[i + 1, 2] - section_z

    # Skip early transient intervals
    if t2 < discard_time:
        continue

    # Root in [t1, t2] if sign changed or one endpoint is exactly on section
    if z1 == 0.0 and z2 == 0.0:
        continue
    if z1 * z2 > 0.0:
        continue

    # Linear interpolation for crossing time (sufficient with small max_step)
    if abs(z2 - z1) < 1e-15:
        alpha = 0.5
    else:
        alpha = -z1 / (z2 - z1)
    alpha = min(1.0, max(0.0, alpha))

    tc = t1 + alpha * (t2 - t1)
    xc, yc, zc = sol.sol(tc)

    # Determine direction using dz/dt at crossing
    dzc = rhs((xc, yc, zc))[2]

    cross_all.append((xc, yc, dzc))
    if dzc > 0:
        cross_up.append((xc, yc))
    else:
        cross_dn.append((xc, yc))

cross_all = np.array(cross_all)
cross_up = np.array(cross_up)
cross_dn = np.array(cross_dn)

print(f"Total crossings (two-sided): {len(cross_all)}")
print(f"Crossings with dz/dt > 0: {len(cross_up)}")
print(f"Crossings with dz/dt < 0: {len(cross_dn)}")

# -------- Plot two-sided section --------
fig_two = go.Figure()

if len(cross_up) > 0:
    fig_two.add_trace(
        go.Scatter(
            x=cross_up[:, 0],
            y=cross_up[:, 1],
            mode="markers",
            marker=dict(size=4, color="royalblue", opacity=0.8),
            name="dz/dt > 0",
        )
    )

if len(cross_dn) > 0:
    fig_two.add_trace(
        go.Scatter(
            x=cross_dn[:, 0],
            y=cross_dn[:, 1],
            mode="markers",
            marker=dict(size=4, color="crimson", opacity=0.8),
            name="dz/dt < 0",
        )
    )

fig_two.add_trace(
    go.Scatter(
        x=static_points[:, 0],
        y=static_points[:, 1],
        mode="markers+text",
        marker=dict(size=10, color=["orange", "green"], symbol="diamond"),
        text=static_labels,
        textposition="top center",
        name="static points",
    )
)

fig_two.update_layout(
    title="Poincare section z=0 (two-sided)",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
)
fig_two.show()

# -------- 3D trajectory + section surface --------
t_plot_start = max(discard_time, t_span[0])
t_plot = np.linspace(t_plot_start, t_span[1], 8000)
traj = sol.sol(t_plot)  # shape: (3, N)

# Section points in 3D (all lie on z = section_z)
if len(cross_all) > 0:
    cross_xyz = np.column_stack([cross_all[:, 0], cross_all[:, 1], np.full(len(cross_all), section_z)])
else:
    cross_xyz = np.empty((0, 3))

# Plane z = section_z over trajectory x/y range
x_min, x_max = float(np.min(traj[0])), float(np.max(traj[0]))
y_min, y_max = float(np.min(traj[1])), float(np.max(traj[1]))
margin_x = 0.08 * (x_max - x_min + 1e-12)
margin_y = 0.08 * (y_max - y_min + 1e-12)

x_plane = np.linspace(x_min - margin_x, x_max + margin_x, 25)
y_plane = np.linspace(y_min - margin_y, y_max + margin_y, 25)
X_plane, Y_plane = np.meshgrid(x_plane, y_plane)
Z_plane = np.full_like(X_plane, section_z)

fig_3d = go.Figure()

fig_3d.add_trace(
    go.Scatter3d(
        x=traj[0],
        y=traj[1],
        z=traj[2],
        mode="lines",
        line=dict(color="black", width=3),
        name="trajectory",
    )
)

fig_3d.add_trace(
    go.Surface(
        x=X_plane,
        y=Y_plane,
        z=Z_plane,
        opacity=0.35,
        colorscale=[[0, "lightblue"], [1, "lightblue"]],
        showscale=False,
        name="z = 0 plane",
    )
)

if len(cross_up) > 0:
    fig_3d.add_trace(
        go.Scatter3d(
            x=cross_up[:, 0],
            y=cross_up[:, 1],
            z=np.full(len(cross_up), section_z),
            mode="markers",
            marker=dict(size=3, color="royalblue"),
            name="dz/dt > 0",
        )
    )

if len(cross_dn) > 0:
    fig_3d.add_trace(
        go.Scatter3d(
            x=cross_dn[:, 0],
            y=cross_dn[:, 1],
            z=np.full(len(cross_dn), section_z),
            mode="markers",
            marker=dict(size=3, color="crimson"),
            name="dz/dt < 0",
        )
    )

fig_3d.add_trace(
    go.Scatter3d(
        x=static_points[:, 0],
        y=static_points[:, 1],
        z=static_points[:, 2],
        mode="markers+text",
        marker=dict(size=6, color=["orange", "green"], symbol="diamond"),
        text=static_labels,
        textposition="top center",
        name="static points",
    )
)

fig_3d.update_layout(
    title="3D attractor with Poincare surface z=0",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="cube",
    ),
    template="plotly_white",
)
fig_3d.show()

Total crossings (two-sided): 177
Crossings with dz/dt > 0: 88
Crossings with dz/dt < 0: 89


### Построить график расхождения двух близлежащих на аттрактора траекторий

In [50]:
# Uses shared imports and system function from the general cell.

# -------- 1) Get a point on attractor (after transient) --------
t_span_pre = (0.0, 200.0)
s0_pre = [0.2, -0.2, 0.1]

sol_pre = solve_ivp(
    f,
    t_span_pre,
    s0_pre,
    rtol=DEFAULT_RTOL,
    atol=DEFAULT_ATOL,
    dense_output=True,
)

if not sol_pre.success:
    raise RuntimeError(f"Pre-integration failed: {sol_pre.message}")

# Base initial condition taken from attractor
t0 = 150.0
s_base = sol_pre.sol(t0)

# -------- 2) Build a close initial condition --------
eps = 1e-8
v = np.array([1.0, -1.0, 0.5], dtype=float)
v = v / np.linalg.norm(v)

s1 = np.array(s_base, dtype=float)
s2 = s1 + eps * v

# -------- 3) Integrate both trajectories on same time grid --------
t_span = (0.0, 400.0)
t_eval = np.linspace(t_span[0], t_span[1], 12000)

sol1 = solve_ivp(f, t_span, s1, t_eval=t_eval, rtol=DEFAULT_RTOL, atol=DEFAULT_ATOL)
sol2 = solve_ivp(f, t_span, s2, t_eval=t_eval, rtol=DEFAULT_RTOL, atol=DEFAULT_ATOL)

if (not sol1.success) or (not sol2.success):
    raise RuntimeError("Integration failed for nearby trajectories")

# -------- 4) Component-wise divergence --------
delta = sol2.y - sol1.y  # shape: (3, N)
dx = np.abs(delta[0])
dy = np.abs(delta[1])
dz = np.abs(delta[2])

# -------- 5) Keep only second part: log-scale plot --------
fig_log = go.Figure()
fig_log.add_trace(
    go.Scatter(
        x=t_eval,
        y=np.maximum(dx, 1e-30),
        mode="lines",
        line=dict(width=2, color="royalblue"),
        name="|dx(t)|",
    )
)
fig_log.add_trace(
    go.Scatter(
        x=t_eval,
        y=np.maximum(dy, 1e-30),
        mode="lines",
        line=dict(width=2, color="seagreen"),
        name="|dy(t)|",
    )
)
fig_log.add_trace(
    go.Scatter(
        x=t_eval,
        y=np.maximum(dz, 1e-30),
        mode="lines",
        line=dict(width=2, color="firebrick"),
        name="|dz(t)|",
    )
)

fig_log.update_layout(
    title="Component-wise divergence of nearby trajectories (log scale)",
    xaxis_title="t",
    yaxis_title="|component difference|",
    yaxis_type="log",
    template="plotly_white",
)
fig_log.show()

print(f"Initial |dx|, |dy|, |dz|: {dx[0]:.3e}, {dy[0]:.3e}, {dz[0]:.3e}")
print(f"Final   |dx|, |dy|, |dz|: {dx[-1]:.3e}, {dy[-1]:.3e}, {dz[-1]:.3e}")

Initial |dx|, |dy|, |dz|: 6.667e-09, 6.667e-09, 3.333e-09
Final   |dx|, |dy|, |dz|: 9.009e-02, 1.042e-01, 2.038e-01


### Найти спектр показателей Ляпунова аттрактора.

In [51]:
# Integrate state + tangent matrix by RK4 and re-orthonormalize each step
def lyapunov_spectrum(s0, dt=0.01, n_steps=120000, burn_in=20000):
    s = np.array(s0, dtype=float)
    Q = np.eye(3)
    sums = np.zeros(3)

    for k in range(n_steps):
        # RK4 for state
        k1 = rhs(s)
        k2 = rhs(s + 0.5 * dt * k1)
        k3 = rhs(s + 0.5 * dt * k2)
        k4 = rhs(s + dt * k3)
        s_new = s + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

        # RK4 for tangent matrix: dQ/dt = J(s)Q
        def g(s_local, Q_local):
            return jacobian(s_local) @ Q_local

        K1 = g(s, Q)
        K2 = g(s + 0.5 * dt * k1, Q + 0.5 * dt * K1)
        K3 = g(s + 0.5 * dt * k2, Q + 0.5 * dt * K2)
        K4 = g(s + dt * k3, Q + dt * K3)
        Q = Q + (dt / 6.0) * (K1 + 2*K2 + 2*K3 + K4)

        # QR step
        Q, R = np.linalg.qr(Q)
        if k >= burn_in:
            sums += np.log(np.maximum(np.abs(np.diag(R)), 1e-30))

        s = s_new

    T = (n_steps - burn_in) * dt
    return sums / T


# Start from attractor point (reuse from previous section if available)
try:
    s_start = np.array(s_base, dtype=float)
except NameError:
    s_start = np.array([0.2, -0.2, 0.1], dtype=float)

lmbd = np.sort(lyapunov_spectrum(s_start))[::-1]
print(f"Lyapunov spectrum: λ1={lmbd[0]:.5f}, λ2={lmbd[1]:.5f}, λ3={lmbd[2]:.5f}")
print(f"Sum λi = {lmbd.sum():.5f}")

Lyapunov spectrum: λ1=0.04873, λ2=0.00188, λ3=-1.05061
Sum λi = -1.00000
